# Notebook 06 — Baseline Evaluation (v3)
**Nhóm 67 | Tuần 4 | Ngôn ngữ Lập trình Python**

Notebook này thực hiện đánh giá **Baseline** bằng cách chạy code chuẩn (AC) của **100 bài toán** để xác nhận độ đúng đắn của dữ liệu và hệ thống máy chấm.

In [7]:
import os
import sys
from pathlib import Path

from pathlib import Path
BASE = next((p for p in [Path.cwd().resolve()] + list(Path.cwd().resolve().parents) if (p / 'data').exists() or (p / 'app.py').exists()), Path.cwd().resolve())

sys.path.insert(0, str(BASE / 'src'))
os.chdir(BASE)

In [3]:
# ═══════════════════════════════════════════════════════════
# PATCH runner_v3.py trực tiếp trên Google Drive
# Thêm so sánh số nguyên lớn vào hàm _compare()
# ═══════════════════════════════════════════════════════════

import os
import sys
from pathlib import Path

# Auto-resolve relative path to Project directory
from pathlib import Path
BASE = next((p for p in [Path.cwd().resolve()] + list(Path.cwd().resolve().parents) if (p / 'data').exists() or (p / 'app.py').exists()), Path.cwd().resolve())

runner_path = str(BASE / 'src' / 'runner_v3.py')

# Đọc nội dung hiện tại
with open(runner_path, "r", encoding="utf-8") as f:
    code = f.read()

# Chuỗi CŨ (dòng so sánh số thực) — cần thêm so sánh int TRƯỚC đó
OLD = "    # Thử so sánh số thực đơn giản\n    try:\n        return math.isclose(float(actual_clean), float(expected_clean), rel_tol=1e-9, abs_tol=1e-9)\n    except (ValueError, TypeError):\n        pass"

# Chuỗi MỚI — thêm khối so sánh int TRƯỚC khi so sánh float
NEW = """    # Thử so sánh số nguyên trực tiếp (bao gồm cả số Bell và số cực lớn)
    try:
        return int(actual_clean) == int(expected_clean)
    except (ValueError, TypeError, OverflowError):
        pass

    # Thử so sánh số thực đơn giản
    try:
        return math.isclose(float(actual_clean), float(expected_clean), rel_tol=1e-9, abs_tol=1e-9)
    except (ValueError, TypeError):
        pass"""

if "int(actual_clean) == int(expected_clean)" in code:
    print("✅ Bản vá đã có sẵn trong runner_v3.py — không cần vá lại!")
elif OLD in code:
    code_patched = code.replace(OLD, NEW)
    with open(runner_path, "w", encoding="utf-8") as f:
        f.write(code_patched)
    print("✅ Đã vá runner_v3.py thành công!")
else:
    print("⚠️ Không tìm thấy đoạn cần vá — hãy paste nội dung dưới đây vào ô tiếp theo để kiểm tra thủ công:")
    # In dòng so sánh thực tế đang có trong file
    for i, line in enumerate(code.split('\n')):
        if 'isclose' in line or 'float(actual' in line:
            print(f"  Dòng {i+1}: {line}")

✅ Bản vá đã có sẵn trong runner_v3.py — không cần vá lại!


In [8]:
import run_baseline

# Chạy baseline và lưu kết quả
run_baseline.main()

  CHẠY BASELINE — Code chuẩn trên 50 bài toán
  Dataset: hidden_v2.json  |  Luồng song song: 4

  ✓ Đã load 100 bài toán
  → Mỗi bài: 3 public + 6 hidden tests

  Bắt đầu chạy song song (4 luồng)...

  [✓] Task  2 is_even                        pub=3/3 hid=6/6 lat=0.0308s
  [✓] Task  1 sum_list                       pub=3/3 hid=6/6 lat=0.0335s
  [✓] Task  4 find_max                       pub=3/3 hid=6/6 lat=0.0364s
  [✓] Task  3 reverse_string                 pub=3/3 hid=6/6 lat=0.0372s
  [✓] Task  5 count_char                     pub=3/3 hid=6/6 lat=0.0315s
  [✓] Task  6 factorial                      pub=3/3 hid=6/6 lat=0.0294s
  [✓] Task  8 remove_duplicates              pub=3/3 hid=6/6 lat=0.0319s
  [✓] Task  7 is_palindrome                  pub=3/3 hid=6/6 lat=0.0319s
  [✓] Task  9 average                        pub=3/3 hid=6/6 lat=0.0359s
  [✓] Task 10 to_uppercase                   pub=3/3 hid=6/6 lat=0.0345s
  [✓] Task 12 sum_even                       pub=3/3 hid=6/6 lat=0.033

In [9]:
import pandas as pd
df_bl = pd.read_csv(BASE / 'results' / 'baseline_summary.csv')

print(f"Số bài toán: {len(df_bl)}")
print(f"Baseline OK (pass 100%): {df_bl['baseline_ok'].sum()}/{len(df_bl)}")
print(f"Latency public trung bình : {df_bl['pub_latency'].mean():.4f}s/test")
print(f"Latency hidden trung bình  : {df_bl['hid_latency'].mean():.4f}s/test")
print(f"Latency tổng trung bình    : {df_bl['avg_latency_total'].mean():.4f}s/test\n")

print("Mẫu bảng kết quả baseline (10 dòng đầu):")
display(df_bl[['task_id', 'func_name', 'topic', 'pub_pass', 'pub_total', 'hid_pass', 'hid_total', 'baseline_ok', 'avg_latency_total']].head(10))

Số bài toán: 100
Baseline OK (pass 100%): 100/100
Latency public trung bình : 0.0339s/test
Latency hidden trung bình  : 0.0338s/test
Latency tổng trung bình    : 0.0338s/test

Mẫu bảng kết quả baseline (10 dòng đầu):


,task_id,func_name,topic,pub_pass,pub_total,hid_pass,hid_total,baseline_ok,avg_latency_total
0,1,sum_list,list,3,3,6,6,True,0.0335
1,2,is_even,math,3,3,6,6,True,0.0308
2,3,reverse_string,string,3,3,6,6,True,0.0372
3,4,find_max,list,3,3,6,6,True,0.0364
4,5,count_char,string,3,3,6,6,True,0.0315
5,6,factorial,math,3,3,6,6,True,0.0294
6,7,is_palindrome,string,3,3,6,6,True,0.0319
7,8,remove_duplicates,list,3,3,6,6,True,0.0319
8,9,average,math,3,3,6,6,True,0.0359
9,10,to_uppercase,string,3,3,6,6,True,0.0345


---
## So sánh Latency theo các bộ test (Set 1 → Set 4)

Phân tích thời gian chấm bài thực tế trên **50 submissions** với 4 cấu hình:
- **Set 1** — 3 public tests
- **Set 2** — 3 public + 6 hidden = 9 tests
- **Set 3** — 3 public + 6–10 hidden = 13 tests (Full-run)
- **Set 4** — Set 3 + **Fail-Fast** (dừng sớm khi gặp lỗi đầu tiên)


In [ ]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

# ── 1. Resolve BASE path ────────────────────────────────────────────────────
BASE = next((p for p in [Path.cwd().resolve()] + list(Path.cwd().resolve().parents) if (p / 'data').exists() or (p / 'app.py').exists()), Path.cwd().resolve())

# ── 2. Load comparison_week4.json ──────────────────────────────────────────
comp_path = BASE / 'results' / 'comparison_week4.json'
with open(comp_path, encoding='utf-8') as f:
    comp = json.load(f)
stats = comp['stats']

SETS   = ['Set 1\n(3 tests)', 'Set 2\n(9 tests)', 'Set 3\n(13 tests)', 'Set 4\n(13+FF)']
KEYS   = ['set1', 'set2', 'set3', 'set4']
COLORS = ['#5C9BD1', '#F29C38', '#3CB371', '#E55B5B']

avg_test = [stats[k]['avg_latency_test_ms']  for k in KEYS]
avg_sub  = [stats[k]['avg_latency_sub_ms']   for k in KEYS]
p95_sub  = [stats[k]['p95_latency_sub_ms']   for k in KEYS]
total_s  = [stats[k]['total_latency_sub_s']  for k in KEYS]

# ── 3. Bảng tóm tắt ────────────────────────────────────────────────────────
df_lat = pd.DataFrame({
    'Bộ test'             : ['Set 1 (3t)', 'Set 2 (9t)', 'Set 3 (13t)', 'Set 4 (13t+FF)'],
    'TB/test (ms)'        : avg_test,
    'TB/submission (ms)'  : avg_sub,
    'P95/submission (ms)' : p95_sub,
    'Tổng 50 bài nộp (s)' : total_s,
})
print('=' * 72)
print('  SO SÁNH LATENCY THEO CÁC BỘ TEST CASES (100 submissions)')
print('=' * 72)
print(df_lat.to_string(index=False))

speedup_total = total_s[2] / total_s[3]
speedup_p95   = p95_sub[2]  / p95_sub[3]
print(f"""
  ✓ Fail-Fast (Set 4) nhanh hơn Full-run (Set 3):
      • Tổng thời gian chấm : {speedup_total:.2f}x  ({total_s[2]:.3f}s → {total_s[3]:.3f}s)
      • Độ trễ P95/bài nộp : {speedup_p95:.2f}x  ({p95_sub[2]:.1f}ms → {p95_sub[3]:.1f}ms)
""")

# ── 4. Vẽ 3 biểu đồ ────────────────────────────────────────────────────────
# SỬA ĐỔI: Thay đổi tỷ lệ figsize (15x6) và sử dụng layout có kiểm tra tương thích
import matplotlib as mpl

fig, axes = plt.subplots(1, 3, figsize=(15, 6), dpi=130)

# Đặt suptitle với top margin để tránh đè lên subplot
fig.suptitle('So sánh Latency theo các Bộ Test (100 submissions)',
             fontsize=14, fontweight='bold', y=1.00)

def style(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_alpha(0.3)
    ax.spines['bottom'].set_alpha(0.3)
    ax.grid(axis='y', alpha=0.2, linestyle='--')

def add_bar_labels(ax, bars, fmt='{:.1f}', suffix=''):
    for bar in bars:
        h = bar.get_height()
        # SỬA ĐỔI: Tăng khoảng cách nhãn lên một chút (3% chiều cao cột) để thoáng hơn
        offset = max(h * 0.03, 0.6)
        ax.text(bar.get_x() + bar.get_width() / 2,
                h + offset,
                fmt.format(h) + suffix,
                ha='center', va='bottom', fontsize=9, fontweight='bold')

# ── Subplot 1: TB latency / test (ms) ──────────────────────────────────────
ax1 = axes[0]
b1 = ax1.bar(SETS, avg_test, color=COLORS, width=0.5, edgecolor='white', linewidth=0.8)
add_bar_labels(ax1, b1, fmt='{:.1f}', suffix=' ms')
ax1.set_title('Latency TB / test case (ms)', fontsize=11, fontweight='bold', pad=12)
ax1.set_ylabel('ms / test', fontsize=9)
ax1.set_ylim(0, max(avg_test) * 1.25 if len(avg_test) and max(avg_test) > 0 else 1)
ax1.tick_params(axis='x', labelsize=8.5)
style(ax1)

# ── Subplot 2: TB & P95 latency / submission (ms) ──────────────────────────
ax2 = axes[1]
x = np.arange(len(SETS))
w = 0.35
b_avg = ax2.bar(x - w / 2, avg_sub, w, color=COLORS, alpha=0.85, edgecolor='white')
b_p95 = ax2.bar(x + w / 2, p95_sub, w, color=COLORS, alpha=0.40,
                edgecolor=[c for c in COLORS], linewidth=1.2)
add_bar_labels(ax2, b_avg, fmt='{:.0f}', suffix=' ms')
add_bar_labels(ax2, b_p95, fmt='{:.0f}', suffix=' ms')
ax2.set_title('Latency TB & P95 / submission (ms)', fontsize=11, fontweight='bold', pad=12)
ax2.set_ylabel('ms / submission', fontsize=9)
# Resolve the true maximum height to avoid clipping average bars (which are larger than P95 due to outliers)
max_val_sub = max(max(avg_sub), max(p95_sub)) if len(avg_sub) and len(p95_sub) else 1
ax2.set_ylim(0, max_val_sub * 1.25)
ax2.set_xticks(x)
ax2.set_xticklabels(SETS, fontsize=8.5)

# SỬA ĐỔI: Điều chỉnh lại tọa độ mũi tên chỉ dòng chữ P95 để không đè lên text của cột
ax2.annotate(
    f'P95: {speedup_p95:.1f}x\nnhanh hơn',
    xy=(x[3] + w / 2, p95_sub[3]),
    xytext=(x[3] - 0.2, p95_sub[3] + max(p95_sub) * 0.35),
    arrowprops=dict(arrowstyle='->', color='#C82323', lw=1.5, connectionstyle="arc3,rad=-0.2"),
    fontsize=8.5, color='#C82323', fontweight='bold', ha='center'
)
avg_patch = mpatches.Patch(color='grey', alpha=0.85, label='TB / bài nộp')
p95_patch = mpatches.Patch(color='grey', alpha=0.40, label='P95 / bài nộp')
ax2.legend(handles=[avg_patch, p95_patch], fontsize=8.5, loc='upper left')
style(ax2)

# ── Subplot 3: Tổng thời gian chấm 100 submissions (s) ──────────────────────
ax3 = axes[2]
b3 = ax3.bar(SETS, total_s, color=COLORS, width=0.5, edgecolor='white', linewidth=0.8)
add_bar_labels(ax3, b3, fmt='{:.3f}', suffix=' s')
ax3.set_title('Tổng thời gian chấm 100 bài nộp (s)', fontsize=11, fontweight='bold', pad=12)
ax3.set_ylabel('Giây (s)', fontsize=9)
ax3.set_ylim(0, max(total_s) * 1.30 if len(total_s) and max(total_s) > 0 else 1)
ax3.tick_params(axis='x', labelsize=8.5)

# SỬA ĐỔI: Đưa box text speedup lên cao hơn đỉnh cột một chút để tránh dính sát vào nhãn số giây
ax3.text(3, total_s[3] + max(total_s) * 0.18,
         f'↑ {speedup_total:.1f}x nhanh hơn\nso với Set 3',
         ha='center', va='bottom', fontsize=8.5, color='#C82323', fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF3CD',
                   alpha=0.9, edgecolor='#F29C38'))
style(ax3)

# Clean layout and save figure properly
plt.tight_layout(rect=[0, 0, 1, 0.95])
out_path = BASE / 'results' / 'baseline_latency_sets.png'
plt.savefig(out_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'✓ Đã lưu biểu đồ: {out_path}')


  SO SÁNH LATENCY THEO CÁC BỘ TEST CASES (100 submissions)
       Bộ test  TB/test (ms)  TB/submission (ms)  P95/submission (ms)  Tổng 50 bài nộp (s)
    Set 1 (3t)        132.31              398.50               185.67              39.8505
    Set 2 (9t)        129.43             1167.28               466.06             116.7278
   Set 3 (13t)        126.94             1285.97               591.76             128.5972
Set 4 (13t+FF)        126.86              170.12               274.45              17.0116

  ✓ Fail-Fast (Set 4) nhanh hơn Full-run (Set 3):
      • Tổng thời gian chấm : 7.56x  (128.597s → 17.012s)
      • Độ trễ P95/bài nộp : 2.16x  (591.8ms → 274.4ms)



AttributeError: 'Figure' object has no attribute '_layout_engine'